In [1]:
import pandas as pd
import numpy as np 
import os 

import gspread
from google.oauth2.service_account import Credentials
from gspread_dataframe import set_with_dataframe

#db
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

#enviorement
from dotenv import load_dotenv
# 1. Cargar las variables de entorno desde el archivo .env
load_dotenv()


True

In [14]:
#engine = get_sql_engine(server=os.getenv("DW_SERVER"), database=os.getenv("DW_DB_Netsuite"))
#Configuración de la API
PATH_CREDS =  os.getenv("GOOGLE_CREDENTIALS")
# Define los alcances (scopes) requeridos
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"]
# Carga las credenciales desde el archivo JSON descargado

creds = Credentials.from_service_account_file(PATH_CREDS, scopes=SCOPES)
client = gspread.authorize(creds)

#https://docs.google.com/spreadsheets/d/1-qkcgiDrKz8Xo21B75GCiJtTm89FCjmqUagy8u_THVg/edit?gid=454498369#gid=454498369

def extract_worksheet(spreadsheet: gspread.Spreadsheet, tab_name: str) -> pd.DataFrame:
    """Lee una pestaña del Google Sheet y la retorna como DataFrame."""
    print("Extrayendo pestaña '%s' del Google Sheet...", tab_name)
    return pd.DataFrame(spreadsheet.worksheet(tab_name).get_all_records())




#Credenciales
ORACLE_USER = os.getenv("ORACLE_USER")
ORACLE_PASS = os.getenv("ORACLE_PASS")
ORACLE_HOST = os.getenv("ORACLE_HOST")
ORACLE_PORT = os.getenv("ORACLE_PORT")
ORACLE_ROLE_ID = os.getenv("ORACLE_ROLE_ID")
ORACLE_ACCOUNT_ID = os.getenv("ORACLE_ACCOUNT_ID")


DEFAULT_SQL_DRIVER = "ODBC Driver 17 for SQL Server"

def get_oracle_conn(host: str,port: str,user: str,password: str,role_id: str,account_id: str,) -> pyodbc.Connection:
    """Retorna una conexión pyodbc a Oracle/NetSuite."""
    try:
        return pyodbc.connect(
            "DRIVER=NetSuite Drivers 64bit;"
            f"Host={host};"
            f"Port={port};"
            "Encrypted=1;"
            "ServerDataSource=NetSuite2.com;"
            "Truststore=system;"
            f"LogonId={user};"
            f"Password={password};"
            "AllowSinglePacketLogout=1;"
            f"CustomProperties=AccountID={account_id};"
            f"RoleID={role_id}"
        )
    except Exception:
        print("Error al conectar con Oracle/NetSuite")


# def get_sql_conn_str(server: str,database: str,user: str | None = None,password: str | None = None,driver: str = DEFAULT_SQL_DRIVER,) -> str:
#     """Genera la cadena de conexión ODBC según el tipo de autenticación recibido."""
#     if user and password:
#         return (
#             f"DRIVER={{{driver}}};"
#             f"SERVER={server};"
#             f"DATABASE={database};"
#             f"UID={user};"
#             f"PWD={password};"
#         )
#     return (
#         f"DRIVER={{{driver}}};"
#         f"SERVER={server};"
#         f"DATABASE={database};"
#         "Trusted_Connection=yes;"
#     )


def get_sql_engine(server: str,database: str,user: str | None = None,password: str | None = None,driver: str = DEFAULT_SQL_DRIVER,) -> Engine:
    """Retorna un Engine de SQLAlchemy listo para Pandas (.to_sql)."""
    driver_encoded = driver.replace(" ", "+")
    if user and password:
        conn_str = f"mssql+pyodbc://{user}:{password}@{server}/{database}?driver={driver_encoded}"
    else:
        conn_str = f"mssql+pyodbc://{server}/{database}?driver={driver_encoded}&trusted_connection=yes"
    try:
        return create_engine(conn_str)
    except Exception:
        print(f"Error al crear Engine SQLAlchemy ({server}/{database})")  # ✅ Correcto



def load_dataframe_to_sql(engine: Engine,df: pd.DataFrame,schema: str,table: str,mode: str = "delete",) -> None:
    """Vacía una tabla del DW (DELETE o TRUNCATE) e inserta el DataFrame, en una sola transacción."""
    tabla_completa = f"{schema}.{table}"
    clear_stmt = "TRUNCATE TABLE" if mode == "truncate" else "DELETE FROM"
    try:
        with engine.begin() as conn:
            conn.execute(text(f"{clear_stmt} {tabla_completa}"))
            df.to_sql(
                schema=schema,
                name=table,
                con=conn,
                if_exists="append",
                index=False,
                chunksize=500,
                method="multi",
            )
    except Exception:
        print("Error durante el ETL hacia '%s' (se ejecutó rollback automático)", tabla_completa)
        


In [ ]:
## Cuentas Contables 

#Abrir conexión de Oracle 
conn_netsuite = get_oracle_conn(
            host=ORACLE_HOST,
            port=ORACLE_PORT,
            user=ORACLE_USER,
            password=ORACLE_PASS,
            role_id=ORACLE_ROLE_ID,
            account_id=ORACLE_ACCOUNT_ID)

t_sql_accounts="""
select *-- a.id Internal_Id, a.acctnumber Number, a.fullname Account, a.isinactive IsInactive
from Account a
--left outer join currency c on a.currency = c.id
where acctnumber is not null and a.id not in (623,630,643)
"""
try:
    # 2. Ejecutar la consulta dentro del try
    df_acct = pd.read_sql(sql=t_sql_accounts, con=conn_netsuite)
    #df_acct["Estado"]=df_acct["IsInactive"].map({"F": "ACTIVO", "T": "INACTIVO"})

finally:
    # 3. Este bloque SIEMPRE se ejecuta (con o sin errores)
    conn_netsuite.close()
    print("Conexión a NetSuite cerrada correctamente.")


C:\Users\luis.meza\AppData\Local\Temp\ipykernel_33632\3507890856.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_acct = pd.read_sql(sql=t_sql_accounts, con=conn_netsuite)


Conexión a NetSuite cerrada correctamente.


In [63]:
df_acct[["id","acctnumber","fullname","isinactive","accttype"]].sample(5).info()

<class 'pandas.DataFrame'>
Index: 5 entries, 398 to 511
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          5 non-null      int64
 1   acctnumber  5 non-null      str  
 2   fullname    5 non-null      str  
 3   isinactive  5 non-null      str  
 4   accttype    5 non-null      str  
dtypes: int64(1), str(4)
memory usage: 240.0 bytes


In [ ]:
#df_balace_leveles=extract_worksheet(spreadsheet=spreadsheet, tab_name="Mapeo")
spreadsheet_id = "1-qkcgiDrKz8Xo21B75GCiJtTm89FCjmqUagy8u_THVg"
spreadsheet = client.open_by_key(spreadsheet_id)
worksheet = spreadsheet.worksheet("Mapeo")
data_range = worksheet.get("A4:H165")
if data_range:
    #headers = data_range[0]  # Primera fila como nombres de columna
    headers= ['Cuenta','Total','L1_Netsuite','L2_Netsuite','L3_Netsuite','L1_Comite','L2_Comite','L3_Comite']
    records = [dict(zip(headers, row)) for row in data_range[1:]]
df_balace_leveles=pd.DataFrame(records)
#extract account's number 
df_balace_leveles["Cuenta"]=df_balace_leveles["Cuenta"].apply(lambda x : x.split("-")[0].strip() )
#remove columns 
df_balace_leveles.drop(columns=["Total"],inplace=True)


In [66]:
df_balace_leveles.head(3)

,Cuenta,L1_Netsuite,L2_Netsuite,L3_Netsuite,L1_Comite,L2_Comite,L3_Comite
0,1001,Activo,Activo Circulante,Efectivo y Equivalentes de Efectivo,NaN,NaN,NaN
1,10100,Activo,Activo Circulante,Efectivo y Equivalentes de Efectivo,ACTIVO,Activo Circulante,Efectivo y Equivalentes de Efectivo
2,10200,Activo,Activo Circulante,Efectivo y Equivalentes de Efectivo,ACTIVO,Activo Circulante,Efectivo y Equivalentes de Efectivo


In [ ]:
#Mapep de Cuentas 
df_MapeoBalance=pd.merge(df_balace_leveles,
         df_acct[["id","acctnumber","fullname","isinactive","accttype"]].rename(columns={"acctnumber":"Cuenta"}),
         how="left",
         on=["Cuenta"]
         )

In [70]:
df_MapeoBalance.sample(5)

,Cuenta,L1_Netsuite,L2_Netsuite,L3_Netsuite,L1_Comite,L2_Comite,L3_Comite,id,fullname,isinactive,accttype
151,29262,Pasivo,Pasivo Circulante,Impuestos por Pagar,PASIVO,Pasivo circulante,Impuestos por Pagar,364,ISR Pag Ret Arrendamiento,F,OthCurrLiab
129,24150,Pasivo,Pasivo Circulante,Préstamos bancarios CP,PASIVO,Pasivo circulante,Préstamos Bancarios CP,1105,Línea De Crédito CP USD,F,OthCurrLiab
14,10272,Activo,Activo Circulante,Efectivo y Equivalentes de Efectivo,ACTIVO,Activo Circulante,Efectivo y Equivalentes de Efectivo,249,ART BNTE 1024464814 USD,F,Bank
105,22321,Pasivo,Pasivo Circulante,Beneficios a los Empleados,PASIVO,Pasivo circulante,Beneficios Directos a los Empleados,306,Reserva PTU,F,OthCurrLiab
118,22334,Pasivo,Pasivo Circulante,Impuestos por Pagar,PASIVO,Pasivo circulante,Impuestos por Pagar,319,Impuesto Sobre Nóminas,F,OthCurrLiab


In [72]:
print(df_MapeoBalance.shape)
print(df_MapeoBalance[["accttype"]].value_counts())

(161, 11)
accttype    
OthCurrLiab     51
Bank            30
FixedAsset      22
OthAsset        16
OthCurrAsset    14
AcctPay         12
AcctRec          7
Equity           5
LongTermLiab     4
Name: count, dtype: int64


In [ ]:
id_balance=",".join(str(x) for x in df_MapeoBalance["id"].to_list())

t_sql_balance=f"""
select 
d.expenseaccount IdCuenta, a.acctnumber Account,
p.periodName Period, TO_NUMBER(TO_CHAR(p.startdate, 'yyyy')) No_Agno , TO_NUMBER(TO_CHAR(p.startdate, 'mm')) No_Mes,
sum(al.amount) Net
from transactionLine d
left outer join transaction t on t.id=d.transaction 
left outer join accountingPeriod p on t.postingperiod=p.id
--AccountingLine
left outer join TransactionAccountingLine al on d.id = al.transactionline
and d.transaction = al.transaction
--detail
left outer join Subsidiary s on d.subsidiary=s.id
left outer join Account a on d.expenseaccount = a.id
where    
d.subsidiary =2  and  --a.acctnumber in ('15400','15420','16300','16320') and 
d.expenseaccount in ({id_balance}) and 
t.posting ='T' --and p.startdate <= TO_DATE('2022/09/30 10:10:00','YYYY/MM/DD HH:MI:SS')
group by  d.expenseaccount , a.acctnumber ,
p.periodName , TO_NUMBER(TO_CHAR(p.startdate, 'yyyy'))  , TO_NUMBER(TO_CHAR(p.startdate, 'mm')) 
--order by No_Agno asc, No_Mes asc
"""
#print(t_sql_balance)

In [83]:
#Abrir conexión de Oracle 
conn_netsuite = get_oracle_conn(host=ORACLE_HOST,port=ORACLE_PORT,user=ORACLE_USER,password=ORACLE_PASS,
            role_id=ORACLE_ROLE_ID,
            account_id=ORACLE_ACCOUNT_ID)
try:
    # 2. Ejecutar la consulta dentro del try
    df_Netsuite = pd.read_sql(sql=t_sql_balance, con=conn_netsuite)

finally:
    # 3. Este bloque SIEMPRE se ejecuta (con o sin errores)
    conn_netsuite.close()
    print("Conexión a NetSuite cerrada correctamente.")


C:\Users\luis.meza\AppData\Local\Temp\ipykernel_33632\1652226265.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Netsuite = pd.read_sql(sql=t_sql_balance, con=conn_netsuite)


Conexión a NetSuite cerrada correctamente.


In [88]:
print(df_Netsuite.sample(2))
print(df_MapeoBalance.sample(2))

      IdCuenta Account    Period No_Agno No_Mes        Net
880        223   10201  sep 2024    2024      9  114621.14
1600       659   19900  dic 2021    2021     12  180049.99
    Cuenta L1_Netsuite           L2_Netsuite          L3_Netsuite L1_Comite  \
145  29232      Pasivo     Pasivo Circulante  Impuestos por Pagar    PASIVO   
75   19101      Activo  Activo no Circulante            Licencias    ACTIVO   

                L2_Comite                  L3_Comite   id  \
145     Pasivo circulante        Impuestos por Pagar  355   
75   Activo no Circulante  Otros Activos Intangibles  286   

                      fullname isinactive     accttype  
145  IVA Pag Ret Arrendamiento          F  OthCurrLiab  
75                   Licencias          F     OthAsset  


In [ ]:
#Balance 
df_Balance= pd.merge(df_Netsuite,
                     df_MapeoBalance[["id","L3_Netsuite","L1_Comite","L2_Comite","L3_Comite"]].rename(columns={"id":"IdCuenta"}), 
                     how="left",
                     on=["IdCuenta"]
                     )

#AgnoMes
df_Balance["AgnoMes"]= df_Balance.apply(lambda x: int(x["No_Agno"])*100 + int(x["No_Mes"]),axis=1)

In [97]:
df_Balance.sample(5)


,IdCuenta,Account,Period,No_Agno,No_Mes,Net,L3_Netsuite,L1_Comite,L2_Comite,L3_Comite,AgnoMes
6229,337,24400,feb 2023,2023,2,-834000.00,Préstamos por pagar a Partes relacionadas,PASIVO,Pasivo circulante,Préstamos por pagar a Partes relacionadas,202302
6322,366,29264,abr 2026,2026,4,5454.11,Impuestos por Pagar,PASIVO,Pasivo circulante,Impuestos por Pagar,202604
3278,667,29220,oct 2024,2024,10,-18095.04,Otras Cuentas por Pagar y Pasivos Acum.,PASIVO,Pasivo circulante,Otras Cuentas por Pagar y Pasivos Acumulados,202410
3060,661,22104,mar 2024,2024,3,-47634.54,Provisiones,PASIVO,Pasivo circulante,Provisiones,202403
3228,1254,16320,abr 2026,2026,4,-57160.38,"Vehículos, neto",ACTIVO,Activo no Circulante,Vehiculos propiedad,202604


In [111]:
df_resumen=df_Balance.query("AgnoMes <= 202606").groupby(['L1_Comite', 'L2_Comite', 'L3_Comite']).agg({'Net': 'sum'}).reset_index(drop=False)
df_resumen=df_resumen.sort_values(by=['L1_Comite', 'L2_Comite', 'L3_Comite'])
df_resumen.to_excel("C:/Users/luis.meza/Desktop/Analista Pricing/Cloude Code Projects/Balance General/Resumen.xlsx",index=False)
# 2. Aplicas el formato de comas a la columna 'Net'
#df_resumen.style.format({'Net': '{:,.2f}'})


### Revisión Montos Balance General con Nueva Estrucutra Server

In [78]:
t_sql="""
select d.expenseaccount IdCuenta,p.startdate,sum(al.amount) Net
from transactionLine d
left outer join transaction t on t.id=d.transaction 
left outer join accountingPeriod p on t.postingperiod=p.id
--AccountingLine
left outer join TransactionAccountingLine al on d.id = al.transactionline and d.transaction = al.transaction
--detail
--left outer join Subsidiary s on d.subsidiary=s.id
left outer join Account a on d.expenseaccount = a.id
where   d.subsidiary =2  and t.posting ='T' 
group by  d.expenseaccount,p.startdate

"""

In [79]:
#Abrir conexión de Oracle 
conn_netsuite = get_oracle_conn(host=ORACLE_HOST,port=ORACLE_PORT,user=ORACLE_USER,password=ORACLE_PASS,
            role_id=ORACLE_ROLE_ID,
            account_id=ORACLE_ACCOUNT_ID)
try:
    # 2. Ejecutar la consulta dentro del try
    df_Balance = pd.read_sql(sql=t_sql, con=conn_netsuite)

finally:
    # 3. Este bloque SIEMPRE se ejecuta (con o sin errores)
    conn_netsuite.close()
    print("Conexión a NetSuite cerrada correctamente.")


C:\Users\luis.meza\AppData\Local\Temp\ipykernel_25804\206200219.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Balance = pd.read_sql(sql=t_sql, con=conn_netsuite)


Conexión a NetSuite cerrada correctamente.


In [80]:
df_Balance.shape

(17232, 3)

In [81]:
conn_DW=get_sql_engine(server= "DW-GOMEX",database="RAW_NS")

t_sql="""
DECLARE @FK_ReporteBalance INT = 4;
DECLARE @FK_ReportePyL INT = 3;
DECLARE @ID_Rubro_ResultadosEjercicio INT = 183;
;With Cuentas as (
--Cuentas Contables + Cuentas Contables de Sueldos y Salarios de la Operación
	select c.*
	from Netsuite.Accounts c
	--Reclasifación contable 
	--https://docs.google.com/spreadsheets/d/1P5ItjEsw3yh8wvev7ZWYop3BomhwEfX7Fdf_UFCLXYU/edit?gid=263435312#gid=263435312
	--pestaña Ideas de la reclasificación
	union all
	select 276000, '15400000', 'Ajuste Activo por derecho de uso vehiculos', 1
	union all
	select 276001, '15400001', 'Ajuste Activo por derecho de uso vehiculos', 1
),
Niveles as (
	select 
	--LEVEL1
	l1.Orden OrdenL1,l1.Nombre_Rubro L1,
	--L2
	DENSE_RANK() OVER (ORDER BY l1.Orden ASC, l1.ID_Rubro, l2.Orden ASC, l2.ID_Rubro) AS OrdenL2,
	l2.Orden PoscicionL2,l2.Nombre_Rubro L2,
	--L3
	DENSE_RANK() OVER (ORDER BY l1.Orden ASC, l1.ID_Rubro, l2.Orden ASC, l2.ID_Rubro,l3.Orden ASC, l3.ID_Rubro) AS OrdenL3,
	l3.Orden PoscicionL3,l3.Nombre_Rubro L3, --l3.ID_Rubro,l3.FK_Reporte,
	--L4
	ROW_NUMBER() OVER (ORDER BY l1.Orden ASC, l2.Orden ASC, l3.Orden,l4.Orden ASC) AS OrdenL4,
	l4.Orden PoscicionL4,l4.Nombre_Rubro L4, l4.ID_Rubro,l4.FK_Reporte
	from Netsuite.Accounts_Levels l4 
	--L3
	join Netsuite.Accounts_Levels l3 on l4.ID_Padre=l3.ID_Rubro and l4.FK_Reporte=l3.FK_Reporte
	--L2
	join Netsuite.Accounts_Levels l2 on l3.ID_Padre=l2.ID_Rubro and l3.FK_Reporte=l2.FK_Reporte
	--L1
	join Netsuite.Accounts_Levels l1 on l2.ID_Padre=l1.ID_Rubro and l2.FK_Reporte=l1.FK_Reporte
	where l4.FK_Reporte=@FK_ReporteBalance and l4.Nivel=4
	--ORDER BY OrdenL1 ASC, OrdenL4 ASC
),
--Mapeo de Cuentas de P&L para el rubro de Resultados del Ejercicio  del balance
MapeoBalance as (
	select m.*
	from Netsuite.Accounts_ReportMapping m
	where m.FK_Reporte=@FK_ReporteBalance  
	union all 
	select  m.Internal_Id, @FK_ReporteBalance FK_Reporte , @ID_Rubro_ResultadosEjercicio FK_PL_Nivel_Base
	from Netsuite.Accounts_ReportMapping m
	where m.FK_Reporte=3--@FK_ReportePyL  
	--Reclasifación contable 
	--https://docs.google.com/spreadsheets/d/1P5ItjEsw3yh8wvev7ZWYop3BomhwEfX7Fdf_UFCLXYU/edit?gid=263435312#gid=263435312
	--pestaña Ideas de la reclasificación
	union all 
	select 276000 Internal_Id, @FK_ReporteBalance FK_Reporte, 143 FK_PL_Nivel_Base
	union all 
	select 276001 Internal_Id, @FK_ReporteBalance FK_Reporte, 145 FK_PL_Nivel_Base
)
--Balance 
select a.Internal_Id,a.Number Account,a.Number+'-'+a.Account [Acc-Name] ,a.Account Description,
n.L1,n.L2,n.L3,n.L4,
n.OrdenL1 [Orden L1],n.OrdenL2 [Orden L2] , n.OrdenL3 [Orden L3],n.OrdenL4 [Orden L4]
from MapeoBalance m
join Cuentas a on m.Internal_Id=a.Internal_Id 
join Niveles n on m.FK_PL_Nivel_Base=n.ID_Rubro and m.FK_Reporte=n.FK_Reporte
"""

with conn_DW.begin() as conn_DW:
    df_ArbolBalance = pd.read_sql(sql=t_sql, con=conn_DW)



In [82]:
conn_DW=get_sql_engine(server= "DW-GOMEX",database="RAW_NS")

#Reclasificación 
t_sql="""
select r.IdCuenta, r.PeriodoContable startdate,r.Monto Net from Netsuite.ReclasificacionesContables r
"""
with conn_DW.begin() as conn_DW:
    df_Reclasif = pd.read_sql(sql=t_sql, con=conn_DW)

df_Reclasif['startdate'] = pd.to_datetime(df_Reclasif['startdate'])


In [83]:
df_back=df_Balance.copy()

In [84]:
print(f"Original Shape{df_Balance.shape}")
df_Balance=pd.concat([df_Balance,df_Reclasif],axis=0)
print(f"New Shape{df_Balance.shape}")


Original Shape(17232, 3)
New Shape(17262, 3)


In [85]:
df_Balance.sample(2)

,IdCuenta,startdate,Net
10182,356.0,2022-08-01,-5162.77
13324,263.0,2023-10-01,0.00


In [86]:
df_ArbolBalance.sample(2)

,Internal_Id,Account,Acc-Name,Description,L1,L2,L3,L4,Orden L1,Orden L2,Orden L3,Orden L4
105,646,15800,15800-Equipo de Cómputo,Equipo de Cómputo,Activos,Activos,Activos Fijos,"Mejoras en locales arrendados, mobiliario y eq...",1,1,3,12
295,1265,40142,40142-Descuento Cuota Telematic,Descuento Cuota Telematic,Capital Contable,Capital Contable,Resultado del ejercicio,Resultado del ejercicio,3,4,13,33


In [ ]:
#Balance 
df_Results= pd.merge(df_Balance,
                     df_ArbolBalance[["Internal_Id","Account","Acc-Name","L1","L2","L3","L4","Orden L1","Orden L2","Orden L3","Orden L4"]].rename(columns={"Internal_Id":"IdCuenta"}), 
                     how="left",
                     on=["IdCuenta"]
                     )

#AgnoMes
#df_Balance["AgnoMes"]= df_Balance.apply(lambda x: int(x["No_Agno"])*100 + int(x["No_Mes"]),axis=1)
df_Results['AgnoMes'] = df_Results['startdate'].dt.year * 100 + df_Results['startdate'].dt.month

# Creación de las 4 nuevas columnas concatenadas
df_Results['L1_Completo'] = df_Results['Orden L1'].astype(str) + '.' + df_Results['L1'].astype(str)
df_Results['L2_Completo'] = df_Results['Orden L2'].astype(str) + '.' + df_Results['L2'].astype(str)
df_Results['L3_Completo'] = df_Results['Orden L3'].astype(str) + '.' + df_Results['L3'].astype(str)
df_Results['L4_Completo'] = df_Results['Orden L4'].astype(str) + '.' + df_Results['L4'].astype(str)

df_Results["NetConver"]=df_Results.apply(lambda x : x["Net"]*(-1) if x ["L1"] in ["Pasivos","Capital Contable"] else x["Net"],axis=1)

In [88]:
df_Results.sample(2)

,IdCuenta,startdate,Net,Account,Acc-Name,L1,L2,L3,L4,Orden L1,Orden L2,Orden L3,Orden L4,AgnoMes,L1_Completo,L2_Completo,L3_Completo,L4_Completo
5117,1288.0,2026-01-01,968216.59,55430,55430-Dep. Vehículos desflotados,Capital Contable,Capital Contable,Resultado del ejercicio,Resultado del ejercicio,3.0,4.0,13.0,33.0,202601,3.0.Capital Contable,4.0.Capital Contable,13.0.Resultado del ejercicio,33.0.Resultado del ejercicio
12409,286.0,2024-11-01,1023061.91,19101,19101-Licencias,Activos,Activos,Activos Fijos,Otros Activos,1.0,1.0,3.0,13.0,202411,1.0.Activos,1.0.Activos,3.0.Activos Fijos,13.0.Otros Activos


In [94]:
print(df_Results.shape)
df_Results.to_excel("C:/Users/luis.meza/Desktop/Balance4Niveles3.xlsx")

(17262, 19)


### Activo por derecho de uso vehiculos

Registros contable manual, disponible en un sheet

In [23]:
#https://docs.google.com/spreadsheets/d/1R_iFnlj-cIOlDPBL0af3fbSIL2a1E47QIRM6ZOkhyYg/edit?gid=781627447#gid=781627447
spreadsheet_id = "1R_iFnlj-cIOlDPBL0af3fbSIL2a1E47QIRM6ZOkhyYg"
spreadsheet = client.open_by_key(spreadsheet_id)
worksheet = spreadsheet.worksheet("data autos propiedad")
data_range = worksheet.get("A:D",value_render_option="UNFORMATTED_VALUE")
if data_range:
    #headers = data_range[0]  # Primera fila como nombres de columna
    headers= ['No_Year','No_Mes','Descripcion','MontoAcumulado']
    records = [dict(zip(headers, row)) for row in data_range[1:]]
df_Vehiculos=pd.DataFrame(records)

#select only specific records and sort values
df_Vehiculos=df_Vehiculos.query("Descripcion=='Activo por derecho de uso vehiculos, neto'").sort_values(by=["No_Year","No_Mes"], ascending=[True,True]).reset_index(drop=True)
#transformaciones 

# .diff() resta el valor actual menos el anterior en una sola operación
df_Vehiculos["Monto"] = df_Vehiculos["MontoAcumulado"].diff()

# Para no perder el valor del primer mes, rellenas el NaN inicial con el valor original acumulado:
df_Vehiculos["Monto"] = df_Vehiculos["Monto"].fillna(
    df_Vehiculos["MontoAcumulado"]
)

#anexar al fecha
df_Vehiculos["PeriodoContable"] = pd.to_datetime(
    df_Vehiculos["No_Year"].astype(str)
    + "-"
    + df_Vehiculos["No_Mes"].astype(str)
    + "-01"  # Forzado el día 01
)

#Valor fijo que corresponda al reporte de balance general disponibles en las tablas de Netsuite.Accounts_Reports
df_Vehiculos["FK_Reporte"]=2

#columns
df_Vehiculos=df_Vehiculos[["FK_Reporte","PeriodoContable","Monto"]]

#layout final

df_Vehiculos=pd.concat(
[df_Vehiculos.assign(IdCuenta=276000, Monto=lambda x: -x["Monto"]),
df_Vehiculos.assign(IdCuenta=276001)
 ], axis=0
).reset_index(drop=True)


In [12]:
#df_Vehiculos.to_excel("C:/Users/luis.meza/Desktop/Analista Pricing/Cloude Code Projects/Balance General/Resumen.xlsx",index=False)

In [ ]:
df_Vehiculos.sample(3)

,FK_Reporte,PeriodoContable,Monto,IdCuenta
18,2,2025-09-01,-4.941015e+07,276001
12,2,2026-05-01,4.942011e+07,276000
1,2,2025-06-01,2.375404e+07,276000


In [31]:
engine = get_sql_engine(server=os.getenv("DW_SERVER"), database=os.getenv("DW_DB_Netsuite"))
load_dataframe_to_sql(engine=engine,df=df_Vehiculos,schema="Netsuite",table="ReclasificacionesContables",mode="delete")